# Tratamento dos Dados

Dados das votações da Câmara dos Deputados de 2019 a 2026 (56ª e 57ª legislaturas), obtidos no portal de [Dados Abertos da Câmara](https://dadosabertos.camara.leg.br/swagger/arquivos.html).

**Objetivo:** montar, para cada legislatura, uma matriz **deputado × votação** com o voto de cada deputado, que será usada no agrupamento, uma tabela com as informações dos deputados (partido, região etc.) e uma tabela com a orientação de cada bancada (Governo, Oposição, partidos) em cada votação, para comparar com os grupos.

**Etapas:**
1. Leitura e junção dos arquivos de cada ano
2. Seleção das colunas úteis
3. Apenas votações do Plenário
4. Apenas votações nominais (quórum de 257 votos)
5. Apenas os votos e orientações dessas votações
6. Transformação do voto em número
7. Criação da matriz deputado × votação
8. Retirada das votações quase unânimes
9. Retirada dos deputados com presença abaixo de 50%
10. Tabela de informações dos deputados
11. Salvamento dos dados tratados

## Download dos dados

In [ ]:
# Código usado para baixar os arquivos do portal de Dados Abertos da Câmara.

import pathlib, requests

PASTA = pathlib.Path("dados/DadosBrutos")
PASTA.mkdir(parents=True, exist_ok=True)
BASE = "https://dadosabertos.camara.leg.br/arquivos"

def baixar(nome, ano=None):
    arquivo = f"{nome}-{ano}.csv" if ano else f"{nome}.csv"
    destino = PASTA / arquivo
    if not destino.exists():
        r = requests.get(f"{BASE}/{nome}/csv/{arquivo}", timeout=180)
        r.raise_for_status()
        destino.write_bytes(r.content)
    return destino

for ano in range(2019, 2027):
    baixar("votacoes", ano)              # lista de votações do ano
    baixar("votacoesVotos", ano)         # voto de cada deputado em cada votação
    baixar("votacoesOrientacoes", ano)   # orientação de cada bancada (inclusive Governo e Oposição) em cada votação
baixar("deputados")                      # cadastro dos deputados

## Bibliotecas

In [2]:
import pandas as pd
import numpy as np

## Dicionário das variáveis

Fonte: [Dados Abertos da Câmara dos Deputados](https://dadosabertos.camara.leg.br/swagger/arquivos.html). Não há dicionário oficial para estes arquivos; as descrições abaixo seguem a [documentação da API](https://dadosabertos.camara.leg.br/swagger/api.html), que usa os mesmos nomes de campos.

### Deputados
Arquivo `deputados.csv` — cadastro de todos os deputados (uma linha por deputado).

| Variável | Descrição |
|---|---|
| `uri` | Link do deputado na API (o ID do deputado é o número no final) |
| `nome` | Nome parlamentar |
| `idLegislaturaInicial` | Primeira legislatura em que o deputado exerceu mandato |
| `idLegislaturaFinal` | Última legislatura em que o deputado exerceu mandato |
| `nomeCivil` | Nome civil completo |
| `cpf` | CPF (geralmente vazio) |
| `siglaSexo` | Sexo (M / F) |
| `urlRedeSocial` | Links das redes sociais |
| `urlWebsite` | Site pessoal |
| `dataNascimento` | Data de nascimento |
| `dataFalecimento` | Data de falecimento (vazio se vivo) |
| `ufNascimento` | UF de nascimento |
| `municipioNascimento` | Município de nascimento |

> Obs.: este arquivo **não tem partido**. O partido vem de `votacoesVotos`.

### Votações
Arquivos `votacoes-AAAA.csv` — uma linha por votação (Plenário e comissões).

| Variável | Descrição |
|---|---|
| `id` | ID da votação (chave para ligar com `votacoesVotos.idVotacao`) |
| `uri` | Link da votação na API |
| `data` | Data da votação |
| `dataHoraRegistro` | Data e hora do registro da votação |
| `idOrgao` / `uriOrgao` | ID e link do órgão onde ocorreu a votação |
| `siglaOrgao` | Sigla do órgão: **PLEN** = Plenário; CCJC, CFFC, etc. = comissões |
| `idEvento` / `uriEvento` | ID e link da sessão/reunião em que ocorreu a votação |
| `aprovacao` | Resultado: 1 = aprovada, 0 = rejeitada |
| `votosSim` | Quantidade de votos "Sim" |
| `votosNao` | Quantidade de votos "Não" |
| `votosOutros` | Quantidade de outros votos (abstenção, obstrução, etc.) |
| `descricao` | Texto descrevendo o resultado da votação |
| `ultimaAberturaVotacao_dataHoraRegistro` | Data e hora da última abertura da votação |
| `ultimaAberturaVotacao_descricao` | Descrição da última abertura da votação |
| `ultimaApresentacaoProposicao_dataHoraRegistro` | Data e hora da apresentação da proposição votada |
| `ultimaApresentacaoProposicao_descricao` | Descrição da proposição votada |
| `ultimaApresentacaoProposicao_idProposicao` | ID da proposição votada |
| `ultimaApresentacaoProposicao_uriProposicao` | Link da proposição votada |

> Obs.: votações simbólicas têm `votosSim = votosNao = votosOutros = 0` e não aparecem em `votacoesVotos`.

### Votos
Arquivos `votacoesVotos-AAAA.csv` — uma linha por voto de um deputado em uma votação nominal.

| Variável | Descrição |
|---|---|
| `idVotacao` | ID da votação (chave para ligar com `votacoes.id`) |
| `uriVotacao` | Link da votação na API |
| `dataHoraVoto` | Data e hora em que o voto foi registrado |
| `voto` | Voto do deputado: **Sim**, **Não**, **Obstrução**, **Abstenção** ou **Artigo 17** |
| `deputado_id` | ID do deputado |
| `deputado_uri` | Link do deputado na API |
| `deputado_nome` | Nome parlamentar |
| `deputado_siglaPartido` | Partido do deputado **no momento do voto** |
| `deputado_uriPartido` | Link do partido na API |
| `deputado_siglaUf` | UF que o deputado representa |
| `deputado_idLegislatura` | Legislatura: 56 = 2019–2022, 57 = 2023–2026 |
| `deputado_urlFoto` | Link da foto do deputado |

> Obs.: **Obstrução** é uma estratégia para derrubar o quórum (na prática, costuma ser um "Não" da oposição). **Artigo 17** é o voto do presidente da Câmara, que não vota (equivale a ausência).

### Orientações
Arquivos `votacoesOrientacoes-AAAA.csv` — uma linha por orientação de uma bancada em uma votação.

| Variável | Descrição |
|---|---|
| `idVotacao` | ID da votação (chave para ligar com `votacoes.id` e `votacoesVotos.idVotacao`) |
| `uriVotacao` | Link da votação na API |
| `siglaOrgao` | Sigla do órgão onde ocorreu a votação (**PLEN** = Plenário) |
| `descricao` | Texto descrevendo o resultado da votação |
| `siglaBancada` | Quem orientou: partidos, blocos (`Bl ...`), federações (`Fdr ...`), **Governo** (também aparece como `GOV.`), **Oposição**, **Maioria** e **Minoria** |
| `uriBancada` | Link da bancada na API (vazio para blocos, Governo, Oposição etc.) |
| `orientacao` | Orientação dada aos deputados: **Sim**, **Não**, **Obstrução**, **Liberado** (cada um vota como quiser) ou vazio (não orientou) |

> Obs.: "Oposição" muda de lado entre as legislaturas: na 56ª é a oposição ao governo Bolsonaro (esquerda) e na 57ª a oposição ao governo Lula (direita).

## 1 - Leitura e junção dos dados

In [3]:
# Votações, votos e orientações vêm em um arquivo por ano. Lemos cada ano, guardamos em uma
# lista e depois juntamos tudo com pd.concat (empilha as linhas das tabelas).
# ignore_index=True refaz o índice de 0 até n, para não repetir índices entre os anos.
pasta = 'dados/DadosBrutos'
anos = range(2019, 2027)   # 2019 até 2026 (o último número não entra)

# Votações: uma linha por votação
lista_votacoes = []
for ano in anos:
    lista_votacoes.append(pd.read_csv(f'{pasta}/votacoes-{ano}.csv', sep=';', encoding='utf-8-sig'))
df_votacoes = pd.concat(lista_votacoes, ignore_index=True)

# Votos: uma linha por voto de um deputado em uma votação
lista_votos = []
for ano in anos:
    lista_votos.append(pd.read_csv(f'{pasta}/votacoesVotos-{ano}.csv', sep=';', encoding='utf-8-sig'))
df_votos = pd.concat(lista_votos, ignore_index=True)

# Orientações: uma linha por orientação de uma bancada (partido, Governo, Oposição...) em uma votação
lista_orientacoes = []
for ano in anos:
    lista_orientacoes.append(pd.read_csv(f'{pasta}/votacoesOrientacoes-{ano}.csv', sep=';', encoding='utf-8-sig'))
df_orientacoes = pd.concat(lista_orientacoes, ignore_index=True)

# Deputados: um arquivo só, com o cadastro de todos os deputados da história da Câmara
df_deputados = pd.read_csv(f'{pasta}/deputados.csv', sep=';', encoding='utf-8-sig')

print(f'Votações:    {df_votacoes.shape}')
print(f'Votos:       {df_votos.shape}')
print(f'Orientações: {df_orientacoes.shape}')
print(f'Deputados:   {df_deputados.shape}')

Votações:    (68433, 20)
Votos:       (1273413, 12)
Orientações: (68838, 7)
Deputados:   (7889, 13)


## 2 - Seleção das colunas úteis

In [4]:
# Mantemos só as colunas que vamos usar.
# O .copy() cria uma tabela nova e independente, o que evita o aviso
# SettingWithCopyWarning quando formos criar colunas novas depois.

# Votações
votacoes_colunas_uteis = ['id', 'data', 'siglaOrgao', 'aprovacao',
                          'votosSim', 'votosNao', 'votosOutros']
df_votacoes = df_votacoes[votacoes_colunas_uteis].copy()

# Votos
votos_colunas_uteis = ['idVotacao', 'voto', 'deputado_id', 'deputado_nome',
                       'deputado_siglaPartido', 'deputado_siglaUf', 'deputado_idLegislatura']
df_votos = df_votos[votos_colunas_uteis].copy()

# Orientações
orientacoes_colunas_uteis = ['idVotacao', 'siglaBancada', 'orientacao']
df_orientacoes = df_orientacoes[orientacoes_colunas_uteis].copy()

# A bancada do governo aparece com dois nomes ('Governo' e 'GOV.'); juntamos em um só
df_orientacoes['siglaBancada'] = df_orientacoes['siglaBancada'].replace({'GOV.': 'Governo'})

# Linhas sem orientação (a bancada não orientou naquela votação) não têm informação
df_orientacoes = df_orientacoes.dropna(subset=['orientacao'])

# Deputados
# O cadastro não tem uma coluna de id: o id é o número no final da uri
# (ex.: '.../api/v2/deputados/204554' -> 204554). Convertemos para número (int)
# para ficar igual à coluna deputado_id da tabela de votos.
df_deputados['deputado_id'] = df_deputados['uri'].str.split('/').str[-1].astype(int)

deputados_colunas_uteis = ['deputado_id', 'nome', 'siglaSexo', 'dataNascimento', 'ufNascimento',
                           'idLegislaturaInicial', 'idLegislaturaFinal']
df_deputados = df_deputados[deputados_colunas_uteis].copy()

In [5]:
print("================= Votações =================")
display(df_votacoes.head(2))
print("================= Votos =================")
display(df_votos.head(2))
print("================= Orientações =================")
display(df_orientacoes.head(2))
print("================= Deputados =================")
display(df_deputados.head(2))

================= Votações =================


,id,data,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros
0,2179189-45,2019-02-12,PLEN,0.0,21,238,32
1,2179189-48,2019-02-12,PLEN,0.0,0,0,0


================= Votos =================


,idVotacao,voto,deputado_id,deputado_nome,deputado_siglaPartido,deputado_siglaUf,deputado_idLegislatura
0,1548586-161,Sim,204521,Abou Anni,UNIÃO,NaN,57
1,1548586-161,Sim,204413,Alcides Rodrigues,PATRIOTA,NaN,57


================= Orientações =================


,idVotacao,siglaBancada,orientacao
0,2165194-103,PCdoB,Sim
1,2165194-103,Governo,Sim


================= Deputados =================


,deputado_id,nome,siglaSexo,dataNascimento,ufNascimento,idLegislaturaInicial,idLegislaturaFinal
0,220593,Abilio Brunini,M,1984-01-31,MT,57,57
1,204379,Acácio Favacho,M,1983-09-28,AP,56,57


## 3 - Apenas votações do Plenário

In [6]:
# Só no Plenário todos os 513 deputados votam. Nas comissões votam apenas os membros
# de cada uma, então os deputados não poderiam ser comparados nas mesmas votações.
print(f'Votações antes:  {df_votacoes.shape}')

df_votacoes = df_votacoes[df_votacoes['siglaOrgao'] == 'PLEN'].copy()

print(f'Votações depois: {df_votacoes.shape}')

Votações antes:  (68433, 7)
Votações depois: (11857, 7)


## 4 - Apenas votações nominais (quórum de 257 votos)

In [7]:
# Muitas votações do Plenário são simbólicas: o resultado é anunciado sem registrar o voto
# de cada deputado (votosSim = votosNao = votosOutros = 0), então não servem para a matriz.
# Uma votação nominal só vale com quórum de maioria absoluta (257 dos 513 deputados),
# por isso mantemos apenas as votações com pelo menos 257 votos registrados.
df_votacoes['votosTotais'] = df_votacoes[['votosSim', 'votosNao', 'votosOutros']].sum(axis=1)

print(f'Votações antes:  {df_votacoes.shape}')

df_votacoes = df_votacoes[df_votacoes['votosTotais'] >= 257].copy()

print(f'Votações depois: {df_votacoes.shape}')

Votações antes:  (11857, 8)
Votações depois: (3099, 8)


## 5 - Apenas os votos e orientações das votações filtradas

In [8]:
# O df_votacoes já foi filtrado (só Plenário e com pelo menos 257 votos).
# Agora aplicamos o mesmo filtro no df_votos e no df_orientacoes: ficam apenas as linhas
# cujo idVotacao está na lista de ids das votações que sobraram.
# .isin() devolve True/False para cada linha, dizendo se o valor está na lista.

# Votos
print(f'Votos antes:  {df_votos.shape}')
df_votos = df_votos[df_votos['idVotacao'].isin(df_votacoes['id'])].copy()
print(f'Votos depois: {df_votos.shape}')

# Orientações
print(f'Orientações antes:  {df_orientacoes.shape}')
df_orientacoes = df_orientacoes[df_orientacoes['idVotacao'].isin(df_votacoes['id'])].copy()
print(f'Orientações depois: {df_orientacoes.shape}')

Votos antes:  (1273413, 7)
Votos depois: (1247362, 7)
Orientações antes:  (62113, 3)
Orientações depois: (55179, 3)


## 6 - Transformação do voto (texto) em número

In [9]:
# Para fazer contas (distâncias, PCA, agrupamento) o voto precisa ser número.
#   Sim       ->  1
#   Não       -> -1
#   Obstrução -> -1  (obstruir é uma forma de se opor, então tratamos como "Não")
#   Abstenção ->  0  (estava presente, mas não se posicionou)
#   Artigo 17 -> NaN (é o presidente da Câmara, que não vota; conta como ausência)
mapa_votos = {'Sim': 1, 'Não': -1, 'Obstrução': -1, 'Abstenção': 0}

# .map() troca cada texto pelo número do dicionário.
# O que não está no dicionário ("Artigo 17") vira NaN automaticamente.
df_votos['voto_num'] = df_votos['voto'].map(mapa_votos)

# Conferindo a conversão: quantas vezes aparece cada par (texto, número)
df_votos[['voto', 'voto_num']].value_counts(dropna=False)

voto       voto_num
Sim         1.0        685864
Não        -1.0        535787
Obstrução  -1.0         18938
Artigo 17   NaN          2799
Abstenção   0.0          2739
NaN         NaN          1235
Name: count, dtype: int64

## 7 - Criação da matriz deputado × votação (uma para cada legislatura)

In [10]:
# Separando os votos de cada legislatura
#   56ª = fev/2019 a jan/2023
#   57ª = fev/2023 a jan/2027
votos_56 = df_votos[df_votos['deputado_idLegislatura'] == 56]
votos_57 = df_votos[df_votos['deputado_idLegislatura'] == 57]

# pivot_table "gira" a tabela de votos:
#   index   -> o que vira LINHA   (cada deputado)
#   columns -> o que vira COLUNA  (cada votação)
#   values  -> o que vai DENTRO de cada célula (o voto em número)
#   aggfunc -> o que fazer se houver mais de um valor na mesma célula.
#              Cada deputado vota só uma vez por votação, então 'first' apenas pega esse voto.
# Se o deputado não votou em uma votação (faltou), a célula fica NaN.

matriz_56 = votos_56.pivot_table(index='deputado_id', columns='idVotacao',
                                 values='voto_num', aggfunc='first')
matriz_57 = votos_57.pivot_table(index='deputado_id', columns='idVotacao',
                                 values='voto_num', aggfunc='first')

# shape = (linhas, colunas) = (deputados, votações)
# .isna().mean().mean() = proporção de células vazias na matriz inteira
print(f'Legislatura 56: {matriz_56.shape} | células vazias: {matriz_56.isna().mean().mean():.0%}')
print(f'Legislatura 57: {matriz_57.shape} | células vazias: {matriz_57.isna().mean().mean():.0%}')

matriz_56.head()

Legislatura 56: (597, 1970) | células vazias: 33%
Legislatura 57: (644, 1126) | células vazias: 37%


idVotacao,105464-319,105464-326,1198512-250,1198512-254,1198512-257,1198512-272,1198512-279,1215346-62,1215346-65,1301128-43,...,946475-127,946475-129,946475-131,946475-138,946494-64,946494-67,946494-68,946494-75,946494-77,946625-101
deputado_id,,,,,,,,,,,,,,,,,,,,,
62881,-1.0,NaN,1.0,-1.0,1.0,1.0,NaN,1.0,1.0,1.0,...,NaN,NaN,1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN
66179,-1.0,-1.0,1.0,-1.0,1.0,1.0,1.0,NaN,NaN,1.0,...,-1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,1.0,1.0,-1.0
66828,-1.0,-1.0,1.0,-1.0,1.0,1.0,1.0,1.0,-1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
67138,NaN,NaN,1.0,-1.0,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0,1.0,1.0,-1.0
68720,-1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,1.0,...,-1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,1.0,1.0,1.0


## 8 - Retirada das votações quase unânimes

In [11]:
# Votações quase unânimes (ex.: 98% "Sim") não ajudam a separar os deputados em grupos,
# porque quase todo mundo votou igual. Vamos retirar as votações em que o lado
# que perdeu (a minoria) teve menos de 10% dos votos Sim/Não.

# --- Legislatura 56 ---
# (matriz == 1) gera True onde o voto foi Sim; .sum() conta os True de cada COLUNA (votação)
sim_56 = (matriz_56 == 1).sum()
nao_56 = (matriz_56 == -1).sum()

# np.minimum pega o menor entre Sim e Não, que é o tamanho da minoria em cada votação
minoria_56 = np.minimum(sim_56, nao_56) / (sim_56 + nao_56)

# .loc[:, condição] -> todas as linhas, e só as colunas em que a condição é True
print(f'Legislatura 56 antes:  {matriz_56.shape}')
matriz_56 = matriz_56.loc[:, minoria_56 >= 0.10]
print(f'Legislatura 56 depois: {matriz_56.shape}')

# --- Legislatura 57 --- (mesma coisa)
sim_57 = (matriz_57 == 1).sum()
nao_57 = (matriz_57 == -1).sum()
minoria_57 = np.minimum(sim_57, nao_57) / (sim_57 + nao_57)

print(f'Legislatura 57 antes:  {matriz_57.shape}')
matriz_57 = matriz_57.loc[:, minoria_57 >= 0.10]
print(f'Legislatura 57 depois: {matriz_57.shape}')

Legislatura 56 antes:  (597, 1970)
Legislatura 56 depois: (597, 1396)
Legislatura 57 antes:  (644, 1126)
Legislatura 57 depois: (644, 894)


## 9 - Retirada dos deputados com presença abaixo de 50%

In [12]:
# Presença = proporção das votações em que o deputado votou.
# .notna() gera True onde a célula tem voto; .mean(axis=1) faz a média de cada LINHA
# (True conta como 1 e False como 0), o que dá a % de presença de cada deputado.
# Isso retira principalmente suplentes que ficaram pouco tempo no cargo e o
# presidente da Câmara (que vota "Artigo 17", que virou NaN).

# --- Legislatura 56 ---
presenca_56 = matriz_56.notna().mean(axis=1)

print(f'Legislatura 56 antes:  {matriz_56.shape}')
matriz_56 = matriz_56[presenca_56 >= 0.5]   # mantém só as linhas (deputados) com presença >= 50%
print(f'Legislatura 56 depois: {matriz_56.shape} | células vazias: {matriz_56.isna().mean().mean():.0%}')

# --- Legislatura 57 --- (mesma coisa)
presenca_57 = matriz_57.notna().mean(axis=1)

print(f'Legislatura 57 antes:  {matriz_57.shape}')
matriz_57 = matriz_57[presenca_57 >= 0.5]
print(f'Legislatura 57 depois: {matriz_57.shape} | células vazias: {matriz_57.isna().mean().mean():.0%}')

Legislatura 56 antes:  (597, 1396)
Legislatura 56 depois: (487, 1396) | células vazias: 21%
Legislatura 57 antes:  (644, 894)
Legislatura 57 depois: (466, 894) | células vazias: 21%


## 10 - Tabela de informações dos deputados

Uma linha por deputado em cada legislatura, com as variáveis que serão comparadas com os grupos (partido, UF, região, sexo, idade, se é novato). Contém apenas os deputados que ficaram nas matrizes.

In [13]:
# Partido e UF vêm da tabela de votos (o cadastro não tem partido).
# groupby junta todas as linhas de cada deputado em cada legislatura, e o .agg diz
# o que calcular para cada grupo, no formato: nova_coluna=('coluna_original', função)
# Alguns deputados trocam de partido durante a legislatura, então usamos o partido
# mais frequente nos votos dele (a moda).

def moda(coluna):
    # .mode() devolve o(s) valor(es) mais frequente(s); [0] pega o primeiro
    return coluna.mode()[0]

df_dep = df_votos.groupby(['deputado_idLegislatura', 'deputado_id']).agg(
    nome=('deputado_nome', 'first'),                  # nome parlamentar
    partido=('deputado_siglaPartido', moda),          # partido mais frequente
    n_partidos=('deputado_siglaPartido', 'nunique'),  # quantos partidos diferentes teve
    uf=('deputado_siglaUf', moda),                    # estado que representa
)

# reset_index transforma legislatura e id (que viraram índice no groupby) de volta em colunas
df_dep = df_dep.reset_index()

df_dep.head()

,deputado_idLegislatura,deputado_id,nome,partido,n_partidos,uf
0,56,62881,Danilo Forte,PSDB,2,CE
1,56,66179,Norma Ayub,DEM,3,ES
2,56,66828,Fausto Pinato,PP,1,SP
3,56,67138,Iracema Portella,PP,1,PI
4,56,68720,Fábio Henrique,PDT,2,SE


In [14]:
# Criando a região a partir da UF, com um dicionário UF -> região
regioes = {'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte',
           'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
           'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste',
           'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
           'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
           'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
           'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'}

df_dep['regiao'] = df_dep['uf'].map(regioes)

df_dep['regiao'].value_counts()

regiao
Sudeste         425
Nordeste        379
Sul             184
Norte           157
Centro-Oeste     96
Name: count, dtype: int64

In [15]:
# Do cadastro (df_deputados) trazemos sexo, data de nascimento e a primeira legislatura.
# O merge liga as duas tabelas pelo deputado_id:
#   how='left'             -> mantém todos os deputados do df_dep, mesmo sem par no cadastro
#   validate='many_to_one' -> confere que cada deputado aparece só uma vez no cadastro
#                             (no df_dep pode aparecer duas vezes: uma em cada legislatura)
# Pegamos só algumas colunas do cadastro para não duplicar a coluna 'nome'.
colunas_cadastro = ['deputado_id', 'siglaSexo', 'dataNascimento', 'idLegislaturaInicial']
df_dep = df_dep.merge(df_deputados[colunas_cadastro], on='deputado_id',
                      how='left', validate='many_to_one')

# Idade no início da legislatura (aproximada pelo ano): a 56ª começa em 2019 e a 57ª em 2023
ano_inicio = df_dep['deputado_idLegislatura'].map({56: 2019, 57: 2023})
df_dep['idade'] = ano_inicio - pd.to_datetime(df_dep['dataNascimento']).dt.year

# Novato = está no primeiro mandato (a primeira legislatura dele é a atual)
df_dep['novato'] = df_dep['idLegislaturaInicial'] == df_dep['deputado_idLegislatura']

In [16]:
# Mantendo só os deputados que ficaram nas matrizes (presença >= 50%),
# para a tabela ter exatamente os mesmos deputados das linhas das matrizes.
# .isin(matriz.index) -> True se o deputado está entre as linhas da matriz
# O símbolo | significa "OU": fica quem está na matriz da 56ª OU na matriz da 57ª.
fica_56 = (df_dep['deputado_idLegislatura'] == 56) & df_dep['deputado_id'].isin(matriz_56.index)
fica_57 = (df_dep['deputado_idLegislatura'] == 57) & df_dep['deputado_id'].isin(matriz_57.index)

print(f'Deputados antes:  {df_dep.shape}')
df_dep = df_dep[fica_56 | fica_57].copy()
print(f'Deputados depois: {df_dep.shape}')

# Ordem final das colunas (dataNascimento e idLegislaturaInicial já foram usadas e saem)
df_dep = df_dep[['deputado_idLegislatura', 'deputado_id', 'nome', 'partido', 'n_partidos',
                 'uf', 'regiao', 'siglaSexo', 'idade', 'novato']]

# Conferindo se sobrou algum valor faltante
print(f'Valores faltantes: {df_dep.isna().sum().sum()}')

df_dep.head()

Deputados antes:  (1241, 12)
Deputados depois: (953, 12)
Valores faltantes: 0


,deputado_idLegislatura,deputado_id,nome,partido,n_partidos,uf,regiao,siglaSexo,idade,novato
1,56,66179,Norma Ayub,DEM,3,ES,Sudeste,F,60,False
2,56,66828,Fausto Pinato,PP,1,SP,Sudeste,M,42,False
4,56,68720,Fábio Henrique,PDT,2,SE,Nordeste,M,47,True
5,56,69871,Bacelar,PODE,2,BA,Nordeste,M,62,False
6,56,72442,Felipe Carreras,PSB,1,PE,Nordeste,M,44,False


## 11 - Salvamento dos dados tratados

In [ ]:
# Salvando as matrizes ainda COM os NaN (faltas).
# O preenchimento dos NaN será feito no notebook de análise, onde vamos comparar métodos diferentes.
# O índice (deputado_id) é salvo junto, como primeira coluna do CSV.
# Para ler depois: pd.read_csv('dados/DadosLimpos/matriz_leg56.csv', index_col=0)
matriz_56.to_csv('dados/DadosLimpos/matriz_leg56.csv')
matriz_57.to_csv('dados/DadosLimpos/matriz_leg57.csv')

# Tabela de deputados, votações do Plenário e orientações das bancadas.
# index=False porque aqui o índice é só 0, 1, 2... e não precisa ser salvo.
df_dep.to_csv('dados/DadosLimpos/deputados_info.csv', index=False)
df_votacoes.to_csv('dados/DadosLimpos/votacoes_plenario.csv', index=False)
df_orientacoes.to_csv('dados/DadosLimpos/orientacoes_plenario.csv', index=False)